# Popularity Baseline

## Load Data

In [1]:
import gdown, zipfile, os

# Skip if already downloaded
if not os.path.exists('dataset/processed'):
    FILE_ID = '1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6'
    gdown.download(id=FILE_ID, output='processed.zip', quiet=False)
    with zipfile.ZipFile('processed.zip', 'r') as z:
        z.extractall('dataset/')
    os.remove('processed.zip')
    print('Downloaded.')
else:
    print('Already exists, skipping download.')

Downloading...
From (original): https://drive.google.com/uc?id=1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6
From (redirected): https://drive.google.com/uc?id=1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6&confirm=t&uuid=1fe56e03-2d71-4412-a6e5-651bc9758e38
To: /content/processed.zip
100%|██████████| 540M/540M [00:02<00:00, 259MB/s]


Downloaded.


In [2]:
import pandas as pd
import numpy as np
import pickle

train_df = pd.read_csv('dataset/processed/train.csv')
val_df   = pd.read_csv('dataset/processed/val.csv')
test_df  = pd.read_csv('dataset/processed/test.csv')
games_df = pd.read_csv('dataset/processed/games_cleaned.csv')

with open('dataset/processed/id_maps.pkl', 'rb') as f:
    maps = pickle.load(f)

n_users = len(maps['user2idx'])
n_items = len(maps['item2idx'])
print(f'n_users: {n_users:,} | n_items: {n_items:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

n_users: 272,184 | n_items: 21,922
Train: 18,151,978 | Val: 272,184 | Test: 272,184


## Popularity Scores

**Bayesian average** - balances average rating with number of ratings, penalising games with few votes

In [3]:
item_counts = train_df.groupby('item_idx')['user_idx'].count().reset_index()
item_counts.columns = ['item_idx', 'rating_count']

item_avg = train_df.groupby('item_idx')['Rating'].mean().reset_index()
item_avg.columns = ['item_idx', 'avg_rating']

popularity_df = item_counts.merge(item_avg, on='item_idx')

C = train_df['Rating'].mean()
m = 50

v = popularity_df['rating_count']
R = popularity_df['avg_rating']
popularity_df['bayesian_avg'] = (v / (v + m)) * R + (m / (v + m)) * C

print(f'Global mean rating C: {C:.4f}')
print(f'Bayesian threshold m: {m}')
print(popularity_df.sort_values('bayesian_avg', ascending=False).head(5))

Global mean rating C: 7.1073
Bayesian threshold m: 50
       item_idx  rating_count  avg_rating  bayesian_avg
21894     21894           533    9.588437      9.375644
21863     21863           350    9.307131      9.032150
18638     18638          3521    9.003190      8.976644
20107     20107           748    9.013074      8.893663
21643     21643          2100    8.925105      8.882830


## Evaluation

Shared evaluation function used by all models. For each user, ranks K candidate items (1 positive test item + 99 random negatives) and computes HR@10, NDCG@10, MRR.

In [4]:
def evaluate(score_series, train_df, test_df, n_negatives=99, K=10, seed=42):
    """
    score_series: pd.Series indexed by item_idx with popularity score
    For each test user: rank 1 positive + n_negatives random unseen items
    """
    rng = np.random.default_rng(seed)
    all_items = np.array(list(maps['item2idx'].values()))

    # Items seen per user in training
    seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

    hits, ndcgs, mrrs = [], [], []

    for _, row in test_df.iterrows():
        uid      = int(row['user_idx'])
        pos_item = int(row['item_idx'])

        # Sample negatives — items user hasn't seen
        user_seen = seen.get(uid, set()) | {pos_item}
        candidates = np.setdiff1d(all_items, list(user_seen))
        neg_items = rng.choice(candidates, size=n_negatives, replace=False)

        # Score and rank all candidates
        items_to_rank = np.append(neg_items, pos_item)
        scores = score_series.reindex(items_to_rank).fillna(0).values
        ranked = items_to_rank[np.argsort(-scores)]

        # Find position of positive item
        pos_rank = np.where(ranked == pos_item)[0][0] + 1  # 1-indexed

        hits.append(1 if pos_rank <= K else 0)
        ndcgs.append(np.log(2) / np.log(pos_rank + 1) if pos_rank <= K else 0)
        mrrs.append(1 / pos_rank if pos_rank <= K else 0)

    return {
        f'HR@{K}':   round(np.mean(hits), 4),
        f'NDCG@{K}': round(np.mean(ndcgs), 4),
        f'MRR@{K}':  round(np.mean(mrrs), 4),
    }

## Results

In [5]:
popularity_df = popularity_df.set_index('item_idx')

print('Evaluating Bayesian average baseline...')
results = evaluate(popularity_df['bayesian_avg'], train_df, test_df)
print(results)

Evaluating Bayesian average baseline...
{'HR@10': np.float64(0.1833), 'NDCG@10': np.float64(0.0946), 'MRR@10': np.float64(0.0679)}


In [6]:
results_df = pd.DataFrame([results], index=['Bayesian popularity'])
print(results_df)

                      HR@10  NDCG@10  MRR@10
Bayesian popularity  0.1833   0.0946  0.0679


## Sample Recommendations

Inspect what the Bayesian average baseline actually recommends for a few users. Since popularity is non-personalized, the ranked list is the same for everyone filtered to items the user hasn't seen.

In [7]:
# Add game names for readability
idx_to_bggid = maps['idx2item']
games_lookup = games_df.set_index('BGGId')[['Name', 'AvgRating', 'BayesAvgRating']]

# Global ranking by Bayesian avg
global_ranking = popularity_df['bayesian_avg'].sort_values(ascending=False)

def get_recommendations(user_idx, train_df, top_k=10):
    seen = set(train_df[train_df['user_idx'] == user_idx]['item_idx'])
    recs = [iid for iid in global_ranking.index if iid not in seen][:top_k]
    bgg_ids = [idx_to_bggid[i] for i in recs]
    return games_lookup.loc[bgg_ids].reset_index()

# Sample 3 users
sample_users = test_df['user_idx'].sample(3, random_state=42).tolist()
for uid in sample_users:
    username = maps['idx2user'][uid]
    print(f'\nTop 10 recommendations for user {username}:')
    print(get_recommendations(uid, train_df).to_string(index=False))


Top 10 recommendations for user VincentParadise:
 BGGId                                  Name  AvgRating  BayesAvgRating
342942                              Ark Nova    8.47839         6.25826
341169  Great Western Trail (Second Edition)    8.45802         6.12471
248562         Mage Knight: Ultimate Edition    8.94937         7.76184
278292               Anachrony: Infinity Box    9.07609         6.63575
324856            The Crew: Mission Deep Sea    8.44932         7.13714
291457          Gloomhaven: Jaws of the Lion    8.69610         8.25163
246900   Eclipse: Second Dawn for the Galaxy    8.67526         7.75460
255984                         Sleeping Gods    8.51042         7.21479
299659 Clash of Cultures: Monumental Edition    8.62371         6.17750
249277                      Brazil: Imperial    8.52007         5.71097

Top 10 recommendations for user Anrab:
 BGGId                                  Name  AvgRating  BayesAvgRating
342942                              Ark Nova  